# 02 — Plan-B / V66 在 CHM13 complex144 上的测试记录

## 面向对象

这份笔记主要写给外部脚本作者阅读。

目标不是评价“脚本好不好”，而是明确：

1. 我们用什么数据测试；
2. 哪些结果来自旧 V4 frontier；
3. 哪些结果来自更新后的 V66 canonical pipeline；
4. 哪些是实现层 open gate，哪些是 formulation / sampler 层现象；
5. 哪些设计值得我们自己的 OLC-QUBO 工作借鉴。

## 1. 测试数据：CHM13 complex144

当前 benchmark：

$$
N=144\text{ reads},
\qquad
M=261\text{ directed candidate edges}.
$$

它不是“几乎唯一解”的 toy graph。

我们已有 exact enumeration：

$$
192
$$

条 Hamilton paths，并且 weighted objective 有唯一 optimum：

$$
W_0=1{,}343{,}093.
$$

runner-up：

$$
1{,}338{,}209.
$$

因此它既能测试：

- 是否能找到 Hamilton path；
- 多个合法 path 中是否能靠近 weighted optimum；
- local move / decomposition 是否会陷入不同 basin。

## 2. 先澄清旧 V4 frontier 中的 $k$

旧 V4 的 frontier state 定义中：

$$
k_{\mathrm{trav}}
=
\text{同时穿过该 window interior 的 traversal 数}.
$$

例如：

$$
\{A\to X\}
$$

是 $k_{\mathrm{trav}}=1$；

$$
\{A\to X,\;B\to Z\}
$$

是 $k_{\mathrm{trav}}=2$。

它和我们 sampler 中的 exchange move size $k_{\mathrm{ex}}$ 完全不是一回事。

旧 V4 的设计是：

- DAG interior 且 $k_{\mathrm{trav}}=1$：经典 DAG DP；
- $k_{\mathrm{trav}}\ge2$：构造多 traversal QUBO / SQA。

## 3. 旧 V4 中实际发现的 frontier-state 问题

在 complex144 的一个典型 window 中，我们观察到：

- window nodes：40；
- interior：35；
- frontier：5；
- ENTRY：2；
- EXIT：3；
- individually reachable ENTRY–EXIT pairs：6。

如果只按 pairwise reachability 枚举 matching，可产生：

$$
1\;(k=0)
+
6\;(k=1)
+
6\;(k=2)
=
13
$$

个 boundary states。

问题出现在 6 个 $k_{\mathrm{trav}}=2$ states。

对某些 state，两条 traversal 单独看分别存在大量路径，但我们做了联合检查后发现：

$$
\boxed{\text{不存在两条同时 vertex-disjoint 的 realization}}
$$

也就是说：

$$
A\leadsto X
$$

单独可达，

$$
B\leadsto Y
$$

也单独可达，

但并不意味着：

$$
\{A\leadsto X,\;B\leadsto Y\}
$$

能同时存在。

旧实现会把这种 state 交给随机 feasible-seed / SQA，最终得到 `UNRESOLVED`。

更合理的状态是：

$$
\boxed{\texttt{PROVEN_INFEASIBLE}}
$$

前提是先做 multi-traversal joint-feasibility precheck。

**这一问题是旧 V4 测试中发现的，不应直接等同于 V66 当前 canonical runtime 的 bug。**

## 4. 对旧 V4 做 joint-feasibility precheck 后的诊断结果

加入仅用于诊断的 DAG $k=2$ vertex-disjoint feasibility precheck 后，旧 frontier/message 思路在 complex144 上能够继续推进。

一次测试中大致得到：

$$
144\text{ active nodes}
\to
31\text{ active nodes},
$$

active edges 约：

$$
261\to86.
$$

过程中：

- $k=1$ states：由 DAG DP 精确处理；
- 一部分 $k=2$ states：预先证明 infeasible；
- 一部分 $k=2$ states：由 sampler 找到 feasible sample。

这个结果支持 frontier/message decomposition 的结构价值，但不代表已经完成 exact global optimization。

## 5. 更新后的 V66：当前 canonical runtime 的边界

更新版 V66 的表示层已经支持更一般的：

- multi-entry；
- multi-exit；
- re-entry；
- multiple traversal；
- closed excursion。

但我们检查到 canonical P08 对 general boundary 仍采取保守策略：

$$
\boxed{\text{GENERAL\_BOUNDARY joint-state solver 尚未 certified}}
$$

因此 single-in / single-out 之外的局部 block 会返回 unresolved，而不是继续用旧 V4 的 $k\ge2$ solver 强行 commit。

从 scientific correctness 角度，这比旧版更安全；但也意味着旧 V4 的 multi-traversal frontier solver 并没有直接成为当前 canonical general-boundary kernel。

## 6. 将 complex144 整体注入 V66 canonical pipeline

为了满足 orientation pairing，我们把 144 个 physical reads 构造成正负两个 orientation：

$$
288\text{ oriented nodes},
\qquad
522\text{ mirrored directed edges}.
$$

正向子图仍保持原：

$$
144/261.
$$

preflight / RC symmetry 检查通过。

未经修改的 canonical full solve 很快进入 `UNRESOLVED`。

测试中发现一个实现集成问题：P18 已把 mate-paired component 分成单 orientation branch，但生成 child state 时 branch tag 与下一层审计期望不一致，因此后续层把 child 当成仍需 RC closure 的完整图检查。

我们做过一个**仅用于诊断的最小 patch**以继续流程；之后仍会遇到前述 GENERAL_BOUNDARY open gate。

因此：

$$
\boxed{\text{目前不能用“full V66 没解出 144”来评价核心算法性能}}
$$

因为 canonical pipeline 还存在未接通接口。

## 7. P20 hidden-DAG decomposition：最值得关注的结果

直接对 complex144 最大的 112-node SCC 做 V66 P20 structural analysis，得到：

$$
112
=
5\text{ active/core}
+
107\text{ hidden DAG}.
$$

在给定 core/interface 决策后，大量 hidden nodes 可以通过 DAG 结构恢复。

一次测试中，P20 找到覆盖：

$$
112/112
$$

节点的内部 path，时间约十几秒量级。

但这里必须注意 V66 当前 primary objective 更偏向：

$$
\max \text{physical-read count},
$$

而不是我们的 weighted-overlap objective：

$$
\max \sum_e w_e.
$$

因此这个结果说明的是：

$$
\boxed{\text{结构可消元性很强}}
$$

而不是“已经找到我们的 weighted optimum”。

另外，$112\to5$ 不能简单解释成“只需 5 个 QUBO variables”。真正复杂度转移到了 boundary/interface state。

## 8. 抽出 V66 cyclic SQA 核心，直接测试 144 图

由于 full pipeline 受 open gate 影响，我们还做了一个更直接的 backend 诊断：

- 绕过完整 decomposition；
- 给 cyclic solver 正确的两个 endpoint；
- 直接把 complex144 的 graph 注入 P31/P33 一类 cyclic backend。

这比真实任务更容易，因为 endpoint 已知。

生成的 QUBO 大约：

$$
2359\text{ variables},
$$

$$
2917\text{ quadratic couplers}.
$$

在小规模 feasible-manifold SQA 参数下，得到的合法 $p\to q$ path 大约覆盖：

$$
115\sim125
$$

个 reads，未达到 144。

因此在当前实现与当前参数下：

$$
\boxed{\text{V66 cyclic backend 尚未达到我们 projected edge-SQA 的 144-read feasibility}}
$$

这不是 universal scaling conclusion，只是 complex144 上的直接 backend 诊断。

## 9. feasible-manifold sampler：值得借鉴与需要澄清的地方

V66 的 proposal 思路很有启发：

1. 在当前 path 上选择两个 anchor；
2. 删除中间 segment；
3. 重新搜索 anchor-to-anchor 替代路径；
4. 一次完成多条 edge 的联合变化。

这和我们最近的 large-neighborhood / reconnect 思路高度相关。

优点：

$$
\boxed{\text{一次 proposal 对应图论 move，而不是单 bit flip}}
$$

并且可以通过较短 path 暂时离开 Hamilton-path manifold，从而有机会跨 basin。

测试中值得进一步讨论的点：

- 随机 DFS proposal 的计算成本可能波动很大，最好加入明确 expansion budget；
- 若目标是“严格 SQA equilibrium sampler”，随机 anchor + DFS 的 proposal 是否对称，需要给出证明；若不对称，应考虑 Hastings ratio；
- 若定位只是 heuristic，则可直接说明，不必强行等同于标准 SQA。

## 10. 给外部脚本作者的具体反馈

### A. Frontier / general boundary

建议把以下两层明确分开：

1. pairwise reachability；
2. multi-traversal joint feasibility。

尤其 $k_{\mathrm{trav}}\ge2$ 时，应在进入 expensive sampler 前尽可能做 vertex/resource-disjoint feasibility screening。

### B. Orientation branch state

建议检查 P18 产生单 orientation child 后，后续 P11/P16 的 branch-tag / RC-closure contract 是否一致。

### C. GENERAL_BOUNDARY

当前保守 unresolved 是合理的，但如果要让 compression 真正覆盖复杂 OLC core，需要补上 certified joint-state solver，或者至少提供明确的 approximate/sample-only channel。

### D. Objective adapter

当前 primary objective 以 read count 为主，而我们的 benchmark 需要 weighted overlap：

$$
\max\sum_e w_e.
$$

建议提供显式 secondary/adapter 接口，避免用户误以为多个 Hamilton paths 会被区分。

### E. Sampler proposal

建议给 path rerouting proposal 增加：

- bounded expansions；
- proposal probability bookkeeping；
- 或明确 heuristic 定位。

## 11. 我们从 V66 借鉴什么

目前最值得借鉴的不是直接复制其 Hamiltonian，而是：

1. **hidden-DAG / structural elimination**：大 SCC 里真正需要联合决策的部分可能远小于 SCC 大小；
2. **frontier/message**：局部消元必须保留 boundary state，而不能只保留单一“局部最佳路径”；
3. **resource signature**：同一 entry/exit 的不同内部实现可能占用不同 physical reads，不能盲目合并；
4. **segment rerouting neighborhood**：图论级 multi-edge move 比裸 bit flip 更符合 OLC 搜索几何；
5. **sampled / validated / certified 分层**：求到可行解与证明 optimum 必须分开。

但目前 100–150 read 规模 complex144 已能被我们不压缩的 projected sampler 直接求解，因此 compression 暂时不是默认第一步。